# Phân loại email spam bằng Decision Tree

Notebook chuyển từ decision_tree.py. Chạy các ô từ trên xuống để làm sạch dữ liệu, huấn luyện, đánh giá và lưu mô hình.


## 1. Import thư viện và thiết lập đường dẫn


In [1]:
import pickle
from pathlib import Path

import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier


In [2]:
cwd = Path.cwd().resolve()
candidates = [cwd, *cwd.parents, cwd / "email_spam_classifier"]
PROJECT_DIR = next(
    (path for path in candidates if (path / "data" / "processed").is_dir()),
    None,
)
if PROJECT_DIR is None:
    raise FileNotFoundError(
        "Không tìm thấy data/processed. Hãy mở notebook từ thư mục dự án hoặc notebooks."
    )
DATA_DIR = PROJECT_DIR / "data" / "processed"
MODEL_PATH = PROJECT_DIR / "models" / "decision_tree.pkl"
print(f"PROJECT_DIR: {PROJECT_DIR}")
print(f"DATA_DIR   : {DATA_DIR}")
print(f"MODEL_PATH : {MODEL_PATH}")

PROJECT_DIR: D:\spam-email-classifier
DATA_DIR   : D:\spam-email-classifier\data\processed
MODEL_PATH : D:\spam-email-classifier\models\decision_tree.pkl


## 2. Các hàm làm sạch dữ liệu


In [3]:
def load_and_clean(path: Path) -> pd.DataFrame:
    """Làm sạch từng tập riêng biệt, không học thống kê từ test."""
    if not path.is_file():
        raise FileNotFoundError(f"Không tìm thấy dữ liệu: {path}")

    df = pd.read_csv(path)
    if not {"text", "label"}.issubset(df.columns):
        raise ValueError(f"{path.name} phải có hai cột text và label.")

    original_count = len(df)
    df = df[["text", "label"]].dropna(subset=["text", "label"]).copy()
    df["text"] = (
        df["text"].astype(str).str.lower()
        .str.replace(r"\s+", " ", regex=True).str.strip()
    )
    df = df.loc[df["text"].ne("")].copy()
    df["label"] = pd.to_numeric(df["label"], errors="raise")
    if not df["label"].isin([0, 1]).all():
        raise ValueError(f"Cột label trong {path.name} phải chỉ có 0 hoặc 1.")
    df["label"] = df["label"].astype(int)
    if df.empty:
        raise ValueError(f"{path.name} không còn mẫu hợp lệ sau làm sạch.")

    print(f"\n{path.name}: {original_count:,} mẫu ban đầu.")
    print(f"Bỏ {original_count - len(df):,} mẫu thiếu nhãn hoặc nội dung rỗng.")
    return df


def clean_training_duplicates(df: pd.DataFrame) -> pd.DataFrame:
    """Loại nội dung mang nhãn mâu thuẫn và mẫu trùng trong train."""
    conflicts = df.groupby("text")["label"].transform("nunique").gt(1)
    print(f"Train: bỏ {int(conflicts.sum()):,} mẫu có nhãn mâu thuẫn.")
    df = df.loc[~conflicts].copy()
    original_count = len(df)
    df = df.drop_duplicates(subset="text").reset_index(drop=True)
    print(f"Train: bỏ {original_count - len(df):,} mẫu trùng nội dung.")
    if set(df["label"].unique()) != {0, 1}:
        raise ValueError("Tập train sau làm sạch phải có cả nhãn 0 và nhãn 1.")
    return df


## 3. Khởi tạo pipeline TF-IDF và Decision Tree


In [4]:
def build_model() -> Pipeline:
    # Chỉ fit TF-IDF trên train; khi predict, test chỉ được transform.
    return Pipeline([
        ("tfidf", TfidfVectorizer(
            max_features=10000,
            min_df=2,
            sublinear_tf=True,
        )),
        ("tree", DecisionTreeClassifier(
            criterion="gini",
            max_depth=20,
            min_samples_leaf=5,
            random_state=42,
        )),
    ])


## 4. Đọc dữ liệu và kiểm tra phân bố nhãn


In [5]:
train_df = clean_training_duplicates(load_and_clean(DATA_DIR / "train.csv"))
test_df = load_and_clean(DATA_DIR / "test.csv")

# Giữ các mẫu hợp lệ trong test, không chia lại dữ liệu.
print("\nSố mẫu theo nhãn (0 = Email thường, 1 = Spam):")
print(pd.DataFrame({
    "train": train_df["label"].value_counts(),
    "test": test_df["label"].value_counts(),
}).reindex([0, 1]).fillna(0).astype(int).to_string())
overlap = int(test_df["text"].isin(set(train_df["text"])).sum())
print(f"Số mẫu test có nội dung xuất hiện trong train: {overlap:,}")
if overlap:
    print("Lưu ý: nội dung trùng giữa train/test có thể làm điểm đánh giá cao hơn thực tế.")



train.csv: 66,755 mẫu ban đầu.
Bỏ 0 mẫu thiếu nhãn hoặc nội dung rỗng.
Train: bỏ 0 mẫu có nhãn mâu thuẫn.
Train: bỏ 249 mẫu trùng nội dung.

test.csv: 16,689 mẫu ban đầu.
Bỏ 0 mẫu thiếu nhãn hoặc nội dung rỗng.

Số mẫu theo nhãn (0 = Email thường, 1 = Spam):
       train  test
label             
0      31437  7907
1      35069  8782
Số mẫu test có nội dung xuất hiện trong train: 144
Lưu ý: nội dung trùng giữa train/test có thể làm điểm đánh giá cao hơn thực tế.


## 5. Huấn luyện mô hình


In [6]:
model = build_model()
print(f"\nĐang huấn luyện trên {len(train_df):,} mẫu train...", flush=True)
model.fit(train_df["text"], train_df["label"])



Đang huấn luyện trên 66,506 mẫu train...


## 6. Đánh giá trên tập test


In [7]:
print(f"Đang đánh giá trên {len(test_df):,} mẫu test...", flush=True)
y_test = test_df["label"]
y_pred = model.predict(test_df["text"])
print(f"\nAccuracy trên tập test: {accuracy_score(y_test, y_pred):.4%}")
print(classification_report(
    y_test, y_pred, labels=[0, 1],
    target_names=["Email thường", "Spam"], digits=4, zero_division=0,
))
print("Ma trận nhầm lẫn (hàng = nhãn thật, cột = nhãn dự đoán):")
print(pd.DataFrame(
    confusion_matrix(y_test, y_pred, labels=[0, 1]),
    index=["Thật: Email thường", "Thật: Spam"],
    columns=["Dự đoán: Email thường", "Dự đoán: Spam"],
).to_string())


Đang đánh giá trên 16,689 mẫu test...

Accuracy trên tập test: 95.0866%
              precision    recall  f1-score   support

Email thường     0.9809    0.9141    0.9463      7907
        Spam     0.9271    0.9839    0.9547      8782

    accuracy                         0.9509     16689
   macro avg     0.9540    0.9490    0.9505     16689
weighted avg     0.9526    0.9509    0.9507     16689

Ma trận nhầm lẫn (hàng = nhãn thật, cột = nhãn dự đoán):
                    Dự đoán: Email thường  Dự đoán: Spam
Thật: Email thường                   7228            679
Thật: Spam                            141           8641


## 7. Lưu mô hình

Ô này lưu cả TF-IDF và cây quyết định vào models/decision_tree.pkl, ghi đè tệp mô hình hiện có.


In [8]:
# Lưu cả TF-IDF và cây quyết định để dùng lại, ghi đè mô hình cũ.
MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)
with MODEL_PATH.open("wb") as file:
    pickle.dump(model, file, protocol=pickle.HIGHEST_PROTOCOL)
print(f"\nĐã lưu mô hình tại: {MODEL_PATH}")



Đã lưu mô hình tại: D:\spam-email-classifier\models\decision_tree.pkl
